# Notebook 01: Exploratory Data Analysis

**Project 04: Predictive Maintenance for Industrial Equipment**  
**STAT 1100 — Langara College — Spring 2026**  
**Team:** Jongmin Lee & Aedrian

This notebook explores the NASA CMAPSS Turbofan Engine Degradation dataset (FD001).
We apply EDA techniques from **Week 5** to understand sensor behavior, identify degradation
patterns, and prepare insights for the data preparation and modeling phases.

**Course Coverage:** Week 2 (Data Sources), Week 5 (EDA & Visualization), Week 8 (PCA), Week 9 (K-Means)

### Setup

**What:** Import all required libraries and set visualization defaults.  
**Why:** Centralizing imports at the top ensures reproducibility and makes dependency requirements clear at a glance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import os
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print(f"pandas: {pd.__version__}, numpy: {np.__version__}, sklearn: {__import__('sklearn').__version__}")

## 1. Data Loading

**What:** Load the NASA CMAPSS FD001 dataset files.  
**Why:** FD001 contains 100 engine run-to-failure trajectories with 21 sensors and 3 operational settings. This is the simplest CMAPSS subset (single operating condition, single fault mode), selected per ADR-001.  
**Course Reference:** Week 2 — Data Collection and Data Sources

In [ ]:
DATA_DIR = 'data/CMAPSSData'

# Column names (no headers in raw files)
columns = ['unit', 'cycle'] + [f'setting_{i}' for i in range(1, 4)] + [f'sensor_{i}' for i in range(1, 22)]

# Load FD001 files — space-separated, no headers
# Note: sep=r'\s+' handles trailing whitespace that creates extra NaN columns
train_df = pd.read_csv(f'{DATA_DIR}/train_FD001.txt', sep=r'\s+', header=None, names=columns)
test_df = pd.read_csv(f'{DATA_DIR}/test_FD001.txt', sep=r'\s+', header=None, names=columns)
rul_df = pd.read_csv(f'{DATA_DIR}/RUL_FD001.txt', sep=r'\s+', header=None, names=['rul'])

print(f"Training set: {train_df.shape[0]:,} rows x {train_df.shape[1]} columns ({train_df['unit'].nunique()} engines)")
print(f"Test set:     {test_df.shape[0]:,} rows x {test_df.shape[1]} columns ({test_df['unit'].nunique()} engines)")
print(f"RUL labels:   {rul_df.shape[0]} engines")

### Dataset Overview

**What:** Display the first rows, data types, missing values, and summary statistics.  
**Why:** A quick sanity check ensures the data loaded correctly and reveals any immediate data quality issues before deeper analysis.

In [ ]:
print("=== First 10 rows ===")
display(train_df.head(10))

print("\n=== Data types ===")
print(train_df.dtypes.value_counts())

print("\n=== Missing values ===")
print(f"Total NaN: {train_df.isna().sum().sum()}")

print("\n=== Summary Statistics ===")
display(train_df.describe().round(2))

## 2. Engine Lifecycle Analysis

**What:** Analyze how long each engine operates before failure.  
**Why:** Understanding lifespan distribution helps us set realistic RUL prediction targets and informs our piecewise-linear cap decision (ADR-002). Engines with very short lifespans contribute fewer training samples.  
**Course Reference:** Week 5 — EDA (distribution analysis, summary statistics)

In [ ]:
lifespans = train_df.groupby('unit')['cycle'].max()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
sns.histplot(lifespans, bins=20, kde=True, ax=axes[0], color='steelblue')
axes[0].axvline(lifespans.mean(), color='red', linestyle='--', label=f'Mean: {lifespans.mean():.0f}')
axes[0].axvline(lifespans.median(), color='orange', linestyle='--', label=f'Median: {lifespans.median():.0f}')
axes[0].set_xlabel('Total Cycles Before Failure')
axes[0].set_ylabel('Number of Engines')
axes[0].set_title('Distribution of Engine Lifespans')
axes[0].legend()

# Box plot
sns.boxplot(x=lifespans, ax=axes[1], color='steelblue')
axes[1].set_xlabel('Total Cycles Before Failure')
axes[1].set_title('Engine Lifespan Box Plot')

plt.tight_layout()
plt.show()

print(f"Min lifespan:    {lifespans.min()} cycles")
print(f"Max lifespan:    {lifespans.max()} cycles")
print(f"Mean lifespan:   {lifespans.mean():.1f} cycles")
print(f"Median lifespan: {lifespans.median():.1f} cycles")
print(f"Std deviation:   {lifespans.std():.1f} cycles")

## 3. Sensor Behavior Over Time

**What:** Visualize all 21 sensor readings over the operational life of a sample engine.  
**Why:** Identifying which sensors show clear degradation trends (increasing/decreasing over time) vs. which remain constant or noisy is essential for feature selection. Constant sensors carry no predictive information and should be removed before modeling.  
**Course Reference:** Week 5 — EDA (visualizing data, identifying patterns)

In [ ]:
# Plot all 21 sensors for engine unit 1
sample_unit = train_df[train_df['unit'] == 1]
sensor_cols = [c for c in train_df.columns if c.startswith('sensor_')]

fig, axes = plt.subplots(7, 3, figsize=(16, 24))
axes = axes.flatten()

for i, sensor in enumerate(sensor_cols):
    axes[i].plot(sample_unit['cycle'], sample_unit[sensor], linewidth=0.8, color='steelblue')
    axes[i].set_title(sensor, fontsize=10)
    axes[i].set_xlabel('Cycle')
    axes[i].tick_params(labelsize=8)

plt.suptitle('Engine Unit 1: All 21 Sensors Over Time', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 4. Multi-Engine Degradation Patterns

**What:** Overlay sensor readings from 5 randomly selected engines to verify degradation consistency.  
**Why:** If sensors show similar degradation trends across different engines, the pattern is systematic (not random noise) and suitable for predictive modeling.  
**Course Reference:** Week 5 — EDA (comparing distributions across groups)

In [ ]:
# Select 5 random engines and plot key sensors that showed degradation trends
np.random.seed(42)
sample_units = np.random.choice(train_df['unit'].unique(), 5, replace=False)

# Sensors known to show degradation in CMAPSS FD001
key_sensors = ['sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_11', 'sensor_15']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, sensor in enumerate(key_sensors):
    for unit in sample_units:
        unit_data = train_df[train_df['unit'] == unit]
        # Normalize x-axis to % of lifespan for comparison
        pct_life = unit_data['cycle'] / unit_data['cycle'].max() * 100
        axes[i].plot(pct_life, unit_data[sensor], linewidth=0.8, alpha=0.7, label=f'Unit {unit}')
    axes[i].set_title(sensor, fontsize=11)
    axes[i].set_xlabel('% of Lifespan')
    axes[i].legend(fontsize=7)

plt.suptitle('5 Engines: Key Sensor Degradation Patterns', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Identifying Constant and Low-Variance Sensors

**What:** Detect sensors with near-zero variance that carry no predictive information.  
**Why:** Constant sensors add noise to models and can cause numerical issues during normalization (division by zero in StandardScaler). They must be identified and removed before preprocessing.  
**Course Reference:** Week 5 — EDA (identifying features to drop), Week 6 — Classification (feature selection before KNN)

In [ ]:
sensor_stats = pd.DataFrame({
    'mean': train_df[sensor_cols].mean(),
    'std': train_df[sensor_cols].std(),
    'min': train_df[sensor_cols].min(),
    'max': train_df[sensor_cols].max(),
    'range': train_df[sensor_cols].max() - train_df[sensor_cols].min()
})

# Flag sensors with very low variance
sensor_stats['constant'] = sensor_stats['std'] < 1e-3

print("=== Sensor Variance Analysis ===")
display(sensor_stats.round(4))

constant_sensors = sensor_stats[sensor_stats['constant']].index.tolist()
useful_sensors = [s for s in sensor_cols if s not in constant_sensors]

print(f"\nConstant/near-constant sensors (will be dropped): {constant_sensors}")
print(f"Useful sensors for modeling ({len(useful_sensors)}): {useful_sensors}")

## 6. Correlation Analysis

**What:** Compute and visualize correlations between useful sensors.  
**Why:** Highly correlated sensors provide redundant information. Understanding correlation structure informs our PCA dimensionality reduction (Week 8) and helps interpret feature importance results.  
**Course Reference:** Week 5 — EDA (correlation heatmaps, `sns.heatmap()`)

In [ ]:
corr_matrix = train_df[useful_sensors].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax)
ax.set_title('Sensor Correlation Matrix (Useful Sensors Only)', fontsize=13)
plt.tight_layout()
plt.show()

# Find strongly correlated pairs
strong_corr = []
for i in range(len(corr_matrix)):
    for j in range(i+1, len(corr_matrix)):
        if abs(corr_matrix.iloc[i, j]) > 0.8:
            strong_corr.append((corr_matrix.index[i], corr_matrix.columns[j], corr_matrix.iloc[i, j]))

print(f"\nStrongly correlated pairs (|r| > 0.8): {len(strong_corr)}")
for s1, s2, r in sorted(strong_corr, key=lambda x: abs(x[2]), reverse=True):
    print(f"  {s1} <-> {s2}: r = {r:.3f}")

## 7. PCA Preview

**What:** Apply PCA to the useful sensor data to understand the intrinsic dimensionality.  
**Why:** With 14 useful sensors, many of which are correlated, PCA can reveal how many independent components capture most of the variance. This preview informs our dimensionality reduction strategy in Notebook 02.  
**Course Reference:** Week 8 — PCA and Factor Analysis (scree plot, explained variance ratio)

In [ ]:
# Standardize before PCA (required for meaningful component extraction)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(train_df[useful_sensors])

pca = PCA()
pca.fit(X_scaled)

# Scree plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Individual variance
axes[0].bar(range(1, len(pca.explained_variance_ratio_) + 1),
            pca.explained_variance_ratio_, color='steelblue', alpha=0.7)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Scree Plot: Individual Variance')

# Cumulative variance
cumvar = np.cumsum(pca.explained_variance_ratio_)
axes[1].plot(range(1, len(cumvar) + 1), cumvar, 'o-', color='steelblue')
axes[1].axhline(y=0.95, color='red', linestyle='--', label='95% threshold')
n_95 = np.argmax(cumvar >= 0.95) + 1
axes[1].axvline(x=n_95, color='orange', linestyle='--', label=f'{n_95} components')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Variance Explained')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Components needed for 95% variance: {n_95}")
print(f"Components needed for 99% variance: {np.argmax(cumvar >= 0.99) + 1}")
print(f"\nVariance per component:")
for i, (var, cum) in enumerate(zip(pca.explained_variance_ratio_, cumvar)):
    print(f"  PC{i+1}: {var:.3f} (cumulative: {cum:.3f})")

## 8. K-Means Engine Clustering

**What:** Cluster sensor readings to identify distinct degradation stages (healthy -> degrading -> critical).  
**Why:** K-Means can reveal natural groupings in the sensor data that correspond to different phases of engine health. This validates our intuition about degradation progression and informs the failure horizon threshold (ADR-005).  
**Course Reference:** Week 9 — Unsupervised Learning - K-Means (elbow method, silhouette score)

In [ ]:
# Use PCA-reduced data for clustering (first 5 components)
X_pca = PCA(n_components=5).fit_transform(X_scaled)

# Elbow method + Silhouette analysis
k_range = range(2, 8)
inertias = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_pca)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_pca, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, inertias, 'o-', color='steelblue')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')

axes[1].plot(k_range, silhouettes, 'o-', color='steelblue')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis')

plt.tight_layout()
plt.show()

# Best k based on silhouette
best_k = list(k_range)[np.argmax(silhouettes)]
print(f"Best k by silhouette score: {best_k} (score: {max(silhouettes):.3f})")

### Cluster Visualization in PCA Space

**What:** Visualize the K-Means clusters (k=3) alongside actual RUL in the first two principal components.  
**Why:** Comparing cluster assignments to true RUL validates whether unsupervised clustering captures meaningful degradation stages without labels.

In [ ]:
# Cluster with k=3 for healthy/degrading/critical interpretation
km_best = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = km_best.fit_predict(X_pca)

# Add RUL for context
max_cycles = train_df.groupby('unit')['cycle'].transform('max')
train_df_temp = train_df.copy()
train_df_temp['rul'] = max_cycles - train_df['cycle']
train_df_temp['cluster'] = cluster_labels

# 2D PCA plot colored by cluster
X_pca_2d = PCA(n_components=2).fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

scatter1 = axes[0].scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=cluster_labels,
                           cmap='viridis', alpha=0.3, s=5)
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].set_title('K-Means Clusters in PCA Space')
plt.colorbar(scatter1, ax=axes[0], label='Cluster')

scatter2 = axes[1].scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=train_df_temp['rul'],
                           cmap='RdYlGn', alpha=0.3, s=5)
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].set_title('Same PCA Space Colored by Actual RUL')
plt.colorbar(scatter2, ax=axes[1], label='RUL (cycles)')

plt.tight_layout()
plt.show()

# Cluster vs RUL summary
print("\n=== Cluster Characteristics ===")
for c in sorted(train_df_temp['cluster'].unique()):
    subset = train_df_temp[train_df_temp['cluster'] == c]
    print(f"Cluster {c}: {len(subset):,} samples, mean RUL = {subset['rul'].mean():.1f}, "
          f"median RUL = {subset['rul'].median():.1f}")

## 9. Operating Condition Analysis

**What:** Check the operational settings to confirm FD001 has a single operating condition.  
**Why:** FD001 should have one operating condition (unlike FD002-FD004). This simplifies our normalization strategy — we can use a single MinMaxScaler instead of per-condition normalization (ADR-006).  
**Course Reference:** Week 3 — Data Wrangling (understanding data structure)

In [ ]:
setting_cols = ['setting_1', 'setting_2', 'setting_3']

print("=== Operating Condition Analysis ===")
for col in setting_cols:
    nunique = train_df[col].nunique()
    print(f"\n{col}: {nunique} unique values")
    if nunique <= 10:
        print(f"  Values: {sorted(train_df[col].unique())}")
    else:
        print(f"  Range: [{train_df[col].min():.4f}, {train_df[col].max():.4f}]")
        print(f"  Std: {train_df[col].std():.6f}")

## 10. Test Set RUL Distribution

**What:** Examine the distribution of true RUL values in the test set.  
**Why:** This shows how many engines in the test set are near failure vs. still healthy at the cutoff point. It helps us understand what our model needs to predict and whether the test set is representative.  
**Course Reference:** Week 5 — EDA (distribution analysis)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(rul_df['rul'], bins=20, kde=True, color='steelblue', ax=ax)
ax.axvline(rul_df['rul'].mean(), color='red', linestyle='--', label=f"Mean: {rul_df['rul'].mean():.1f}")
ax.axvline(30, color='orange', linestyle='--', label='Failure horizon (30 cycles)')
ax.set_xlabel('Remaining Useful Life (cycles)')
ax.set_ylabel('Number of Engines')
ax.set_title('Test Set: True RUL Distribution at Cutoff Point')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Test set RUL statistics:")
print(f"  Min: {rul_df['rul'].min()}")
print(f"  Max: {rul_df['rul'].max()}")
print(f"  Mean: {rul_df['rul'].mean():.1f}")
print(f"  Engines with RUL <= 30 (critical): {(rul_df['rul'] <= 30).sum()}/{len(rul_df)}")
print(f"  Engines with RUL > 130 (healthy):  {(rul_df['rul'] > 130).sum()}/{len(rul_df)}")

## 11. Key Findings Summary

Based on our exploratory analysis:

1. **Dataset structure:** 100 engines, average lifespan ~206 cycles, ranging from 128 to 362 cycles
2. **Constant sensors (will be dropped):** sensors 1, 5, 6, 10, 16, 18, 19 — zero or near-zero variance
3. **Informative sensors:** sensors 2, 3, 4, 7, 8, 11, 12, 13, 14, 15, 17, 20, 21 show clear degradation trends
4. **High correlations:** Several sensor pairs are strongly correlated (|r| > 0.8), supporting PCA dimensionality reduction
5. **PCA:** First ~5 components capture >95% of variance — significant dimensionality reduction possible
6. **K-Means:** 3 clusters map roughly to healthy/degrading/critical engine states, validating degradation progression
7. **Single operating condition:** FD001 settings are effectively constant, simplifying normalization
8. **Test set RUL:** Ranges widely, with some engines near failure and others still healthy

**Next steps (Notebook 02):** Data preparation — RUL label construction, normalization, feature engineering, and train/test splitting.